# L05. Neural Network: Feed Forward Basics

---

## 학습 목표

이 노트북을 마치면 다음을 할 수 있습니다.

1. **activation function**의 역할을 말할 수 있다.
2. `nn.Linear`가 하는 계산을 shape와 함께 설명할 수 있다.
3. 간단한 **DNN(MLP)** 모델을 직접 정의할 수 있다.
4. MNIST / FashionMNIST 같은 공개 데이터셋을 불러와 모델에 넣을 수 있다.
5. 아직 학습 전이라도 **forward pass 결과(logit, probability, prediction)** 를 읽을 수 있다.


## 1. 왜 neural network가 필요한가?

Linear classifier는 입력을 한 번에 점수로 바꿉니다.

$$
score = Wx + b
$$

그런데 현실의 데이터는 훨씬 더 복잡합니다. 그래서 우리는 중간에 **hidden layer**를 두고,
각 층마다 **activation function**을 넣어서 더 복잡한 표현을 만들 수 있게 합니다.

구조는 다음과 같습니다.

```
image → flatten → linear → activation → linear → output
```

**모델의 구조와 데이터 흐름**을 먼저 확인합니다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

np.random.seed(42)
torch.manual_seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('NumPy version  :', np.__version__)
print('PyTorch version:', torch.__version__)


## 2. Activation Function

Activation function은 입력값을 다음 층으로 넘기기 전에 일정한 규칙으로 변환하는 함수입니다.

- 입력값 하나를 넣으면
- 규칙에 따라 값을 조금 바꾸고
- 그 결과를 다음 층으로 넘긴다.

대표 예시는 ReLU입니다.

$$
ReLU(x) = \max(0, x)
$$

즉, 음수는 0으로 만들고 양수는 그대로 둡니다.


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def tanh(x):
    return np.tanh(x)


def relu(x):
    return np.maximum(0, x)


x = np.linspace(-5, 5, 400)
plt.figure(figsize=(8, 4))
plt.plot(x, sigmoid(x), label='Sigmoid', linewidth=2)
plt.plot(x, tanh(x), label='Tanh', linewidth=2)
plt.plot(x, relu(x), label='ReLU', linewidth=2)
plt.axhline(0, color='gray', linewidth=0.8)
plt.axvline(0, color='gray', linewidth=0.8)
plt.title('Activation Functions')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
values = np.array([-2.0, -0.5, 0.0, 0.5, 2.0])
print('input   :', values)
print('sigmoid :', np.round(sigmoid(values), 4))
print('tanh    :', np.round(tanh(values), 4))
print('relu    :', np.round(relu(values), 4))


## 3. `nn.Linear`는 정확히 무엇을 할까?

PyTorch의 `nn.Linear(in_features, out_features)`는 다음 계산을 합니다.

$$
out = xW^T + b
$$

**shape**를 함께 확인해보겠습니다.

- 입력이 `(batch_size, in_features)`
- 출력이 `(batch_size, out_features)`

즉, 샘플 여러 개를 한 번에 처리합니다.


In [ ]:
linear = nn.Linear(3, 4)
example_x = torch.tensor([
    [0.2, 0.7, 1.0],
    [1.2, -0.3, 0.5],
], dtype=torch.float32)
example_out = linear(example_x)

print('input shape :', tuple(example_x.shape))
print('weight shape:', tuple(linear.weight.shape))
print('bias shape  :', tuple(linear.bias.shape))
print('output shape:', tuple(example_out.shape))
print(example_out)


## 4. DNN 모델 만들기

가장 기본적인 **MLP(Multi-Layer Perceptron)** 를 만듭니다.

구조는 다음과 같습니다.

1. 이미지를 1차원으로 펼친다.
2. hidden layer 1개를 지난다.
3. activation을 적용한다.
4. 출력층에서 10개 클래스 점수를 만든다.

먼저 hidden layer 1개인 구조부터 보겠습니다.


In [ ]:
class BasicDNN(nn.Module):
    def __init__(self, input_dim=28 * 28, hidden_dim=128, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x


model = BasicDNN()
print(model)


## 5. 공개 데이터셋 불러오기

`torchvision.datasets`가 제공하는 **MNIST**를 사용합니다.

- 손글씨 숫자 데이터셋
- 크기: `28 x 28`
- 클래스: `0 ~ 9`

이 데이터셋은 딥러닝 입문에서 가장 많이 쓰이는 기본 예제 중 하나입니다.


In [ ]:
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

images, labels = next(iter(train_loader))
print('images shape:', tuple(images.shape))
print('labels      :', labels.tolist())


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
axes = axes.ravel()

for i, ax in enumerate(axes):
    ax.imshow(images[i, 0], cmap='gray')
    ax.set_title(f'label={labels[i].item()}')
    ax.axis('off')

plt.tight_layout()
plt.show()


## 6. Forward Pass 해보기

이제 이미지를 모델에 넣고 출력값을 확인합니다.

출력은 아직 확률이 아니라 **logit(score)** 입니다.
필요하면 softmax를 적용해서 확률처럼 볼 수 있습니다.


In [ ]:
with torch.no_grad():
    logits = model(images)
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)

print('logits shape:', tuple(logits.shape))
print('predictions :', preds.tolist())
print('true labels :', labels.tolist())
print()
print('first sample probability vector:')
print(torch.round(probs[0] * 1000) / 1000)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(9, 4))
axes = axes.ravel()

for i, ax in enumerate(axes):
    ax.imshow(images[i, 0], cmap='gray')
    ax.set_title(f'true={labels[i].item()}\npred={preds[i].item()}')
    ax.axis('off')

plt.tight_layout()
plt.show()


## 7. Hidden Dimension을 바꾸면?

모델의 hidden size는 대표적인 하이퍼파라미터입니다.

아래 셀은 hidden dimension만 바꿔서 출력 shape가 어떻게 유지되는지 확인하는 예제입니다.


In [ ]:
for hidden_dim in [32, 64, 128, 256]:
    temp_model = BasicDNN(hidden_dim=hidden_dim)
    with torch.no_grad():
        temp_logits = temp_model(images)
    print(f'hidden_dim={hidden_dim:3d} -> output shape={tuple(temp_logits.shape)}')


## 8. FashionMNIST에도 같은 모델 넣어보기

숫자 대신 옷 이미지 데이터셋인 **FashionMNIST**도 구조가 거의 같습니다.

- 이미지 크기: `28 x 28`
- 클래스 수: 10

즉, 같은 DNN 구조를 거의 그대로 사용할 수 있습니다.


In [ ]:
fashion_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
fashion_loader = DataLoader(fashion_dataset, batch_size=8, shuffle=True)
fashion_images, fashion_labels = next(iter(fashion_loader))

with torch.no_grad():
    fashion_logits = model(fashion_images)
    fashion_preds = fashion_logits.argmax(dim=1)

print('fashion batch shape:', tuple(fashion_images.shape))
print('fashion labels     :', fashion_labels.tolist())
print('fashion preds      :', fashion_preds.tolist())


## 9. 핵심 정리

- Neural Network는 **Linear + Activation**을 반복해서 만든다.
- activation이 없으면 깊게 쌓아도 비선형 패턴을 잘 표현할 수 없다.
- `nn.Linear`는 입력을 다른 차원의 벡터로 바꾸는 층이다.
- `nn.Flatten()`은 이미지를 DNN에 넣기 위해 일렬로 펴는 역할을 한다.
- 여기서는 **학습(training)** 보다 **모델 구조와 forward pass**를 먼저 확인했다.

→ 다음 토픽: **Backpropagation & Optimization**


## 10. 연습 문제

### 📝 Exercise 1

ReLU, Sigmoid, Tanh 중 하나를 골라서 다음을 직접 설명해보세요.

- 그래프 모양은 어떤가?
- 출력 범위는 무엇인가?
- 입력이 음수일 때 어떤 일이 일어나는가?

### 📝 Exercise 2

`BasicDNN`의 hidden dimension을 `64`, `256`으로 바꿔서 forward pass를 다시 실행해보세요.

- 출력 shape는 어떻게 되는가?
- 파라미터 수는 어떻게 달라질 것 같은가?

### 📝 Exercise 3

`BasicDNN`의 activation을 `ReLU` 대신 `Sigmoid` 또는 `Tanh`로 바꿔보세요.

- 코드에서 어느 줄을 수정하면 되는가?
- 출력값의 분위기가 어떻게 달라지는가?

### 📝 Exercise 4

MNIST 대신 FashionMNIST 배치를 넣어 같은 모델의 출력을 확인해보세요.

- 입력 shape는 같은가?
- 정답 클래스의 의미는 어떻게 다른가?

### 📝 Exercise 5

hidden layer를 하나 더 추가한 `DeeperDNN`을 만들어보세요.

힌트:

```python
self.fc1 = nn.Linear(28 * 28, 128)
self.fc2 = nn.Linear(128, 64)
self.fc3 = nn.Linear(64, 10)
```

forward에서는 activation을 층 사이마다 넣어보세요.
